# Hand Keypoint Extraction — MediaPipe
## Khmer Finger Spelling Dataset

This notebook extracts **21 hand keypoints (X, Y, Z) per hand** from gesture images using **MediaPipe Hands**.

### Output format (per row in CSV)
```
image_name, label, handedness, x0 y0 z0 x1 y1 z1 ... x20 y20 z20 (Right hand — 63 values), x0 y0 z0 ... x20 y20 z20 (Left hand — 63 values)
```

- **Fixed-length vector**: 126 keypoint values total (63 per hand) + 1 handedness column
- **Column 1** → `image_name`
- **Column 2** → `label` (class folder name)
- **Column 3** → `handedness` — which hands were detected: `right`, `left`, or `both`
- **Column 4** → Right hand keypoints (63 values); zeros if no right hand detected
- **Column 5** → Left hand keypoints (63 values); zeros if no left hand detected
- **X, Y**: normalised to [0, 1] relative to image dimensions
- **Z**: depth relative to the wrist landmark (negative = closer to camera); not normalised to image dimensions
- **No detection** → image copied to `bad_images/`, skipped from CSV
- **Resume-safe**: re-running skips already-processed images

> **Handedness note**: MediaPipe determines Left/Right from the image as-is (non-mirrored view).
> If your images were captured with a **front-facing (selfie/mirrored) camera**, the Left/Right labels will be swapped — flip the image horizontally before extracting if that is your case.

### Supported Categories

| Category | Extraction | Notes |
|---|---|---|
| Consonants | All classes, all slots | Two-level: `ClassName/Main` and `ClassName/Sub` |
| Independent Vowels | All classes | Single-level folder |
| Khmer Numbers | All classes | Single-level folder |
| No Action | Single class | Entire Raw Dataset folder = one class |
| Vowels / General | Customizable list | Edit `VOWEL_CLASSES` in Cell 3 (empty = all) |

### Workflow
1. Edit **Cell 3** — set source/output root paths and the `VOWEL_CLASSES` list
2. Run **Cell 4** to preview all discovered jobs
3. Run all cells top-to-bottom — Cell 9 processes every job in sequence


In [1]:
# ============================================================
#  CELL 2 — IMPORTS
# ============================================================
import os
import cv2
import csv
import shutil
import logging
import urllib.request
import numpy as np
import mediapipe as mp

# mediapipe 0.10+ Tasks API (replaces the removed mp.solutions)
BaseOptions          = mp.tasks.BaseOptions
HandLandmarker       = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode    = mp.tasks.vision.RunningMode

from pathlib import Path
from datetime import datetime

print("All libraries imported successfully.")
print(f"OpenCV version  : {cv2.__version__}")
print(f"MediaPipe version: {mp.__version__}")

All libraries imported successfully.
OpenCV version  : 4.11.0
MediaPipe version: 0.10.35


In [2]:

# ============================================================
#  CELL 3 — MULTI-CLASS EXTRACTION CONFIGURATION
#  Edit the ROOT PATHS and VOWEL_CLASSES sections below.
# ============================================================

# ---- MediaPipe Model -----------------------------------------------
MODEL_PATH = r"E:\Test-extract-keypoint-body-hand\hand_landmarker.task"
MODEL_URL  = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"

if not os.path.exists(MODEL_PATH):
    print(f"Downloading hand landmarker model → {MODEL_PATH} ...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print("Download complete.")
else:
    print(f"Model already present: {MODEL_PATH}")

# ---- MediaPipe Hands settings --------------------------------------
MAX_NUM_HANDS            = 2    # Max hands to detect per image (1 or 2)
MIN_DETECTION_CONFIDENCE = 0.4  # Hand detection confidence threshold
MIN_PRESENCE_CONFIDENCE  = 0.4  # Hand presence confidence threshold
MIN_TRACKING_CONFIDENCE  = 0.5  # Tracking threshold (unused in IMAGE mode)

# ---- Resize (disabled — images are passed at their original size) --
RESIZE_WIDTH  = None   # Set to an int (e.g. 640) to enable resizing
RESIZE_HEIGHT = None   # Set to an int (e.g. 360) to enable resizing

# ---- Progress & supported formats ----------------------------------
PROGRESS_EVERY   = 500
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tiff'}

# ==================================================================
#  SOURCE ROOTS — Edit these paths to match your folder layout
#  Two consonant roots are supported:
#    CONSONANT_RAW_ROOT          → augmented images (no suffix)
#    CONSONANT_ORIGINAL_RAW_ROOT → original images  (label gets _original suffix)
#  Both roots feed into the SAME output CSV file per class/slot.
# ==================================================================

# CON_RAW_ROOT = r"E:\Keypoint Extraction\Raw Dataset Consonant"
# IND_VOWEL_RAW_ROOT  = r"E:\Keypoint Extraction\Raw Dataset Independent Vowels"
# KH_NUMBER_RAW_ROOT  = r"E:\Keypoint Extraction\Raw Dataset Khmer Numbers"
# NO_ACTION_RAW_ROOT  = r"E:\Keypoint Extraction\Raw Dataset No Action"
VOWEL_RAW_ROOT      = r"E:\Keypoint Extraction\៍"

# ==================================================================
#  OUTPUT ROOTS — Edit these paths to where you want the results
# ==================================================================

# # Consonants
# CONSONANT_CSV_ROOT  = r"E:\Keypoint Extraction\Raw Dataset Consonant\Keypoint"
# CONSONANT_BAD_ROOT  = r"E:\Keypoint Extraction\Raw Dataset Consonant\Bad Images"
# CONSONANT_LOG_ROOT  = r"E:\Keypoint Extraction\Raw Dataset Consonant\Log"

# # Independent Vowels
# IND_VOWEL_CSV_ROOT  = r"E:\Keypoint Extraction\Raw Dataset Independent Vowels\Keypoint"
# IND_VOWEL_BAD_ROOT  = r"E:\Keypoint Extraction\Raw Dataset Independent Vowels\Bad Images"
# IND_VOWEL_LOG_ROOT  = r"E:\Keypoint Extraction\Raw Dataset Independent Vowels\Log"

# # Khmer Numbers
# KH_NUMBER_CSV_ROOT  = r"E:\Keypoint Extraction\Raw Dataset Khmer Numbers\Keypoint"
# KH_NUMBER_BAD_ROOT  = r"E:\Keypoint Extraction\Raw Dataset Khmer Numbers\Bad Images"
# KH_NUMBER_LOG_ROOT  = r"E:\Keypoint Extraction\Raw Dataset Khmer Numbers\Log"

# # No Action
# NO_ACTION_CSV_ROOT  = r"E:\Keypoint Extraction\Raw Dataset No Action\Keypoint"
# NO_ACTION_BAD_ROOT  = r"E:\Keypoint Extraction\Raw Dataset No Action\Bad Images"
# NO_ACTION_LOG_ROOT  = r"E:\Keypoint Extraction\Raw Dataset No Action\Log"

# Vowels / General
VOWEL_CSV_ROOT      = r"E:\Keypoint Extraction\៍\Keypoint"
VOWEL_BAD_ROOT      = r"E:\Keypoint Extraction\៍\Bad Images"
VOWEL_LOG_ROOT      = r"E:\Keypoint Extraction\៍\Log"

# ==================================================================
#  CLASS NAME OVERRIDES
#  Edit this mapping to rename labels in the extracted output.
#  Keys can be either the exact folder path or the current label.
#  Values are the class names you want in the CSV and output filenames.
# ==================================================================
CLASS_NAME_MAP = {

}

# ==================================================================
#  CLASS SELECTION
#  Only these class folders will be extracted.
#  Leave the list empty ( [] ) to extract ALL classes automatically.
# ==================================================================
VOWEL_CLASSES = []   # Empty = extract every subfolder in VOWEL_RAW_ROOT

print("Configuration loaded.")
print(f"  Raw Dataset root : {VOWEL_RAW_ROOT}")
print(f"  Keypoint output  : {VOWEL_CSV_ROOT}")
print(f"  Bad images folder: {VOWEL_BAD_ROOT}")
print(f"  Log folder       : {VOWEL_LOG_ROOT}")
print(f"  Classes selected : {'ALL (auto-discover)' if not VOWEL_CLASSES else len(VOWEL_CLASSES)}")
print(f"  Label overrides  : {'NONE (using folder names)' if not CLASS_NAME_MAP else len(CLASS_NAME_MAP)}")


Model already present: E:\Test-extract-keypoint-body-hand\hand_landmarker.task
Configuration loaded.
  Raw Dataset root : E:\Keypoint Extraction\៍
  Keypoint output  : E:\Keypoint Extraction\៍\Keypoint
  Bad images folder: E:\Keypoint Extraction\៍\Bad Images
  Log folder       : E:\Keypoint Extraction\៍\Log
  Classes selected : ALL (auto-discover)
  Label overrides  : NONE (using folder names)


In [3]:
# ============================================================
#  CELL 4 — JOB BUILDER
#  Auto-discovers all extraction jobs from the configured roots.
#  Run this after Cell 3 to preview the full job list.
# ============================================================

def build_jobs():
    """
    Discover and return all extraction jobs as a list of dicts.
    Each dict has keys: input_folder, label, output_csv,
                        bad_images_folder, log_file.

    Category rules
    ──────────────
    Consonants (augmented)  — CONSONANT_RAW_ROOT  (direct image folder)
                              label derived from last two path parts: ClassName_Slot
                              e.g. path .../ក/Main  → label = ក_Main
    Consonants (original)   — CONSONANT_ORIGINAL_RAW_ROOT  (direct image folder)
                              label = ClassName_Slot_original
                              output_csv = same file as augmented counterpart
    Independent Vowels      — IND_VOWEL_RAW_ROOT/ClassName  (only if variable defined)
    Khmer Numbers           — KH_NUMBER_RAW_ROOT/ClassName  (only if variable defined)
    No Action               — NO_ACTION_RAW_ROOT itself      (only if variable defined)
    Vowels / General        — VOWEL_RAW_ROOT/ClassName       (only if variable defined)

    All optional category roots are silently skipped when their
    variables are not defined (e.g. commented out in Cell 3).
    Class name overrides are read from CLASS_NAME_MAP in Cell 3.
    """
    jobs = []

    label_overrides_raw = globals().get('CLASS_NAME_MAP', {}) or {}
    label_overrides = {}
    for key, value in label_overrides_raw.items():
        label_overrides[os.path.normpath(str(key))] = value

    def resolve_label(default_label, source_path=None):
        """Return a custom label when one is defined for the path or label."""
        if source_path is not None:
            normalized_path = os.path.normpath(str(source_path))
            if normalized_path in label_overrides:
                return label_overrides[normalized_path]
        if default_label in label_overrides:
            return label_overrides[default_label]
        return default_label

    # ---- Helper: one job for a flat image folder --------------------
    def _make_consonant_job(raw_root, label_suffix, warn_key):
        """
        Create one extraction job treating raw_root as a direct image folder.
        base_label is derived from the last two path components (ClassName_Slot),
        e.g.  .../ក/Main  →  base_label = 'ក_Main'.
        The output CSV uses base_label only (no suffix) so augmented and
        original images are written to the SAME CSV file.
        """
        if not os.path.isdir(raw_root):
            print(f"[WARNING] {warn_key} not found: {raw_root}")
            return None
        p = Path(raw_root)
        base_label = f"{p.parent.name}_{p.name}"   # e.g.  ក_Main
        resolved_base_label = resolve_label(base_label, raw_root)
        label = f"{resolved_base_label}{label_suffix}"  # e.g.  ក_Main_original
        return {
            "input_folder":      raw_root,
            "label":             label,
            # Both roots share the same CSV (resolved base label, no suffix in filename)
            "output_csv":        os.path.join(CONSONANT_CSV_ROOT,
                                              f"{resolved_base_label}_keypoints.csv"),
            "bad_images_folder": os.path.join(CONSONANT_BAD_ROOT, label),
            "log_file":          os.path.join(CONSONANT_LOG_ROOT, f"{label}.log"),
        }

    # ---- CONSONANTS (augmented) — no label suffix -------------------
    _cn_raw = globals().get('CONSONANT_RAW_ROOT')
    if _cn_raw:
        job = _make_consonant_job(_cn_raw, "", "Consonant augmented")
        if job:
            jobs.append(job)

    # ---- CONSONANTS (original) — _original suffix, same CSV ---------
    _cn_orig_raw = globals().get('CONSONANT_ORIGINAL_RAW_ROOT')
    if _cn_orig_raw:
        job = _make_consonant_job(_cn_orig_raw, "_original", "Consonant original")
        if job:
            jobs.append(job)

    # ---- INDEPENDENT VOWELS: single-level — ClassName ---------------
    # Silently skipped when variable is not defined in Cell 3.
    _iv_raw = globals().get('IND_VOWEL_RAW_ROOT')
    _iv_csv = globals().get('IND_VOWEL_CSV_ROOT')
    _iv_bad = globals().get('IND_VOWEL_BAD_ROOT')
    _iv_log = globals().get('IND_VOWEL_LOG_ROOT')
    if _iv_raw:
        if os.path.isdir(_iv_raw):
            for class_name in sorted(os.listdir(_iv_raw)):
                class_path = os.path.join(_iv_raw, class_name)
                if not os.path.isdir(class_path):
                    continue
                resolved_class_name = resolve_label(class_name, class_path)
                jobs.append({
                    "input_folder":      class_path,
                    "label":             resolved_class_name,
                    "output_csv":        os.path.join(_iv_csv, f"{resolved_class_name}_keypoints.csv"),
                    "bad_images_folder": os.path.join(_iv_bad, resolved_class_name),
                    "log_file":          os.path.join(_iv_log, f"{resolved_class_name}.log"),
                })
        else:
            print(f"[WARNING] Independent Vowels root not found: {_iv_raw}")

    # ---- KHMER NUMBERS: single-level — ClassName --------------------
    _kn_raw = globals().get('KH_NUMBER_RAW_ROOT')
    _kn_csv = globals().get('KH_NUMBER_CSV_ROOT')
    _kn_bad = globals().get('KH_NUMBER_BAD_ROOT')
    _kn_log = globals().get('KH_NUMBER_LOG_ROOT')
    if _kn_raw:
        if os.path.isdir(_kn_raw):
            for class_name in sorted(os.listdir(_kn_raw)):
                class_path = os.path.join(_kn_raw, class_name)
                if not os.path.isdir(class_path):
                    continue
                resolved_class_name = resolve_label(class_name, class_path)
                jobs.append({
                    "input_folder":      class_path,
                    "label":             resolved_class_name,
                    "output_csv":        os.path.join(_kn_csv, f"{resolved_class_name}_keypoints.csv"),
                    "bad_images_folder": os.path.join(_kn_bad, resolved_class_name),
                    "log_file":          os.path.join(_kn_log, f"{resolved_class_name}.log"),
                })
        else:
            print(f"[WARNING] Khmer Numbers root not found: {_kn_raw}")

    # ---- NO ACTION: entire Raw Dataset folder = one class -----------
    _na_raw = globals().get('NO_ACTION_RAW_ROOT')
    _na_csv = globals().get('NO_ACTION_CSV_ROOT')
    _na_bad = globals().get('NO_ACTION_BAD_ROOT')
    _na_log = globals().get('NO_ACTION_LOG_ROOT')
    if _na_raw:
        if os.path.isdir(_na_raw):
            resolved_no_action = resolve_label("No_Action", _na_raw)
            jobs.append({
                "input_folder":      _na_raw,
                "label":             resolved_no_action,
                "output_csv":        os.path.join(_na_csv, f"{resolved_no_action}_keypoints.csv"),
                "bad_images_folder": os.path.join(_na_bad, resolved_no_action),
                "log_file":          os.path.join(_na_log, f"{resolved_no_action}.log"),
            })
        else:
            print(f"[WARNING] No Action root not found: {_na_raw}")

    # ---- VOWELS / GENERAL: flat root or class subfolders ---------------
    _vw_raw = globals().get('VOWEL_RAW_ROOT')
    _vw_csv = globals().get('VOWEL_CSV_ROOT')
    _vw_bad = globals().get('VOWEL_BAD_ROOT')
    _vw_log = globals().get('VOWEL_LOG_ROOT')
    if _vw_raw:
        if os.path.isdir(_vw_raw):
            entries = sorted(os.listdir(_vw_raw))
            image_files = [name for name in entries if Path(name).suffix.lower() in IMAGE_EXTENSIONS]
            subfolders = [name for name in entries if os.path.isdir(os.path.join(_vw_raw, name))]

            # Flat folder of images: treat the whole root as one extraction job.
            if image_files:
                root_label = os.path.basename(os.path.normpath(_vw_raw))
                resolved_root_label = resolve_label(root_label, _vw_raw)
                jobs.append({
                    "input_folder":      _vw_raw,
                    "label":             resolved_root_label,
                    "output_csv":        os.path.join(_vw_csv, f"{resolved_root_label}_keypoints.csv"),
                    "bad_images_folder": os.path.join(_vw_bad, resolved_root_label),
                    "log_file":          os.path.join(_vw_log, f"{resolved_root_label}.log"),
                })

            # Folder-of-folders: extract only the requested class subfolders.
            if subfolders:
                classes = VOWEL_CLASSES if VOWEL_CLASSES else subfolders
                for class_name in classes:
                    class_path = os.path.join(_vw_raw, class_name)
                    if not os.path.isdir(class_path):
                        print(f"  [SKIP] Class folder not found: {class_path}")
                        continue
                    resolved_class_name = resolve_label(class_name, class_path)
                    jobs.append({
                        "input_folder":      class_path,
                        "label":             resolved_class_name,
                        "output_csv":        os.path.join(_vw_csv, f"{resolved_class_name}_keypoints.csv"),
                        "bad_images_folder": os.path.join(_vw_bad, resolved_class_name),
                        "log_file":          os.path.join(_vw_log, f"{resolved_class_name}.log"),
                    })
        else:
            print(f"[WARNING] Raw Dataset root not found: {_vw_raw}")

    return jobs


# ---- Preview -------------------------------------------------------
EXTRACTION_JOBS = build_jobs()
print(f"\nTotal extraction jobs discovered: {len(EXTRACTION_JOBS)}")
print(f"\n  {'#':<5}  {'Label':<40}  {'Output CSV'}")
print("  " + "-" * 90)
for i, job in enumerate(EXTRACTION_JOBS, start=1):
    csv_name = os.path.basename(job["output_csv"])
    print(f"  {i:<5}  {job['label']:<40}  {csv_name}")



Total extraction jobs discovered: 1

  #      Label                                     Output CSV
  ------------------------------------------------------------------------------------------
  1      ៍                                         ៍_keypoints.csv


In [4]:
# ============================================================
#  CELL 5 — HELPER / UTILITY FUNCTIONS
# ============================================================

def setup_logging(log_file):
    """
    Configure a logger that writes DEBUG+ to the log file
    and INFO+ to the console.
    Safe to call multiple times (clears duplicate handlers).
    """
    logger = logging.getLogger("KeypointExtractor")
    logger.setLevel(logging.DEBUG)

    # Remove stale handlers from previous notebook cell re-runs
    if logger.handlers:
        logger.handlers.clear()

    fmt = logging.Formatter(
        "%(asctime)s [%(levelname)s] %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    # File handler — writes everything (DEBUG and above)
    log_dir = os.path.dirname(log_file)
    if log_dir:
        os.makedirs(log_dir, exist_ok=True)
    fh = logging.FileHandler(log_file, encoding="utf-8")
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(fmt)

    # Console handler — INFO and above only (less noise)
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    ch.setFormatter(fmt)

    logger.addHandler(fh)
    logger.addHandler(ch)
    return logger


def setup_output_dirs(output_csv, bad_images_folder):
    """Create the output CSV directory and the bad_images folder."""
    csv_dir = os.path.dirname(output_csv)
    if csv_dir:
        os.makedirs(csv_dir, exist_ok=True)
    os.makedirs(bad_images_folder, exist_ok=True)


def load_processed_images(output_csv):
    """
    Read the existing output CSV and return a set of image names
    that have already been processed.

    This is the resume mechanism: any image whose name appears in
    the CSV will be skipped on subsequent runs.
    """
    processed = set()
    if not os.path.exists(output_csv):
        return processed
    try:
        with open(output_csv, "r", encoding="utf-8") as f:
            reader = csv.reader(f)
            for row in reader:
                if row:
                    processed.add(row[0].strip())
    except Exception as exc:
        print(f"[WARNING] Could not read existing CSV for resume: {exc}")
    return processed


def get_image_files(folder, extensions):
    """
    Return a sorted list of absolute image paths found in `folder`.
    Only files whose extension (lowercase) is in `extensions` are included.
    """
    files = [
        os.path.join(folder, name)
        for name in os.listdir(folder)
        if Path(name).suffix.lower() in extensions
    ]
    return sorted(files)


print("Helper functions defined.")


Helper functions defined.


In [5]:
# ============================================================
#  CELL 6 — KEYPOINT EXTRACTION HELPERS
# ============================================================

def format_hand_keypoints(hand_landmark_list):
    """
    Extract normalised (x, y, z) coordinates for all 21 landmarks
    of a single detected hand.

    mediapipe 0.10 Tasks API returns hand_landmark_list as a plain
    list of NormalizedLandmark objects (not wrapped in .landmark).
      - .x and .y are normalised to [0, 1] (image dimensions).
      - .z is depth relative to the wrist landmark (negative = closer
        to camera); not normalised to image dimensions.

    Returns:
        list[float] — 63 values: [x0, y0, z0, x1, y1, z1, ..., x20, y20, z20]
    """
    coords = []
    for lm in hand_landmark_list:       # iterate directly — no .landmark needed
        coords.append(round(float(lm.x), 6))
        coords.append(round(float(lm.y), 6))
        coords.append(round(float(lm.z), 6))
    return coords  # always length 63


def zero_hand_keypoints():
    """
    Return 63 zeros representing a missing / undetected hand.
    Used to pad single-hand images to a fixed-length vector.
    """
    return [0.0] * 63  # 21 keypoints × 3 (x, y, z) = 63 values


def enhance_image(image_bgr):
    """
    Apply CLAHE (Contrast Limited Adaptive Histogram Equalization)
    to improve contrast on dark, shadowy, or low-contrast images.

    How it works:
      - Converts to LAB colour space (separates brightness from colour).
      - Applies CLAHE only to the L (lightness) channel.
      - Converts back to BGR — colours are unchanged, only contrast is improved.

    This helps MediaPipe detect hands in difficult images without
    affecting keypoint coordinate accuracy once detection succeeds.

    Args:
        image_bgr : NumPy array in BGR colour order.

    Returns:
        NumPy array in BGR colour order with enhanced contrast.
    """
    lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    # clipLimit: max contrast amplification (2.0 is a good balance)
    # tileGridSize: divides image into tiles for local contrast — (8,8) is standard
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_enhanced = clahe.apply(l)

    lab_enhanced = cv2.merge([l_enhanced, a, b])
    return cv2.cvtColor(lab_enhanced, cv2.COLOR_LAB2BGR)


def preprocess_image(image_bgr, resize_w, resize_h):
    """
    Optionally resize the image for faster MediaPipe inference,
    then convert from BGR (OpenCV default) to RGB (MediaPipe requirement).

    Args:
        image_bgr  : NumPy array in BGR colour order (from cv2.imdecode).
        resize_w   : Target width in pixels, or None to skip resizing.
        resize_h   : Target height in pixels, or None to skip resizing.

    Returns:
        NumPy array in RGB colour order, optionally resized.
    """
    if resize_w and resize_h:
        image_bgr = cv2.resize(
            image_bgr,
            (resize_w, resize_h),
            interpolation=cv2.INTER_AREA,  # best quality for downscaling
        )
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)


print("Keypoint extraction helpers defined.")
print("  format_hand_keypoints → 63 values per hand (x, y, z × 21 landmarks)")
print("  zero_hand_keypoints   → 63 zeros for missing hand")


Keypoint extraction helpers defined.
  format_hand_keypoints → 63 values per hand (x, y, z × 21 landmarks)
  zero_hand_keypoints   → 63 zeros for missing hand


In [6]:
# ============================================================
#  CELL 7 — SINGLE-IMAGE PROCESSING FUNCTION
# ============================================================

def process_image(image_path, detector, logger,
                  resize_w, resize_h, bad_images_folder):
    """
    Process one image end-to-end using the mediapipe 0.10 Tasks API:
      1. Load image with OpenCV (Unicode-safe path handling).
      2. Try detection on the original image first.
      3. If no hand found, retry with CLAHE enhancement as a fallback.
      4. Extract keypoints with handedness-based slot assignment.
      5. Return structured result, or None on failure.

    Slot assignment (consistent across all images):
      - Slot 1 (right_kps) → Right hand keypoints, or 63 zeros if absent
      - Slot 2 (left_kps)  → Left hand keypoints,  or 63 zeros if absent

    Handedness values:
      - "right"  → only right hand detected
      - "left"   → only left hand detected
      - "both"   → both hands detected
    """
    image_name = os.path.basename(image_path)

    # --- Step 1: Load image (Unicode-safe) ----------------------------
    try:
        raw = np.fromfile(image_path, dtype=np.uint8)
        image_bgr = cv2.imdecode(raw, cv2.IMREAD_COLOR)
    except Exception as exc:
        logger.warning(f"Could not read image — skipped: {image_name} ({exc})")
        return None

    if image_bgr is None:
        logger.warning(f"Unreadable/corrupted image — skipped: {image_name}")
        return None

    # --- Step 2: First attempt — original image -----------------------
    image_rgb = preprocess_image(image_bgr, resize_w, resize_h)
    mp_image  = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
    result    = detector.detect(mp_image)

    # --- Step 3: Fallback — retry with CLAHE enhancement --------------
    # Only applied when the first attempt fails.
    # Avoids over-processing already clear images (which can hurt detection).
    if not result.hand_landmarks:
        logger.debug(f"No detection on original — retrying with enhancement: {image_name}")
        enhanced_bgr = enhance_image(image_bgr)
        image_rgb    = preprocess_image(enhanced_bgr, resize_w, resize_h)
        mp_image     = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
        result       = detector.detect(mp_image)

    # --- Step 4: Handle no-detection after both attempts --------------
    if not result.hand_landmarks:
        logger.debug(f"No hand detected — copying to bad_images: {image_name}")
        dest_path = os.path.join(bad_images_folder, image_name)
        if not os.path.exists(dest_path):
            shutil.copy2(image_path, dest_path)
        return None

    # --- Step 5: Extract keypoints with handedness-based slot assignment ---
    # result.handedness is a list parallel to result.hand_landmarks.
    # Each element is a list of Category objects; [0].category_name is 'Right' or 'Left'.
    # We build a dict keyed by side so the slot is always deterministic
    # regardless of the order MediaPipe returns the hands.
    #
    #   Slot 1 (right_kps) → Right hand  (zeros if not detected)
    #   Slot 2 (left_kps)  → Left hand   (zeros if not detected)
    hand_map = {}
    for handedness_list, landmarks in zip(result.handedness, result.hand_landmarks):
        side = handedness_list[0].category_name   # 'Right' or 'Left'
        hand_map[side] = landmarks

    right_kps = (format_hand_keypoints(hand_map['Right'])
                 if 'Right' in hand_map else zero_hand_keypoints())
    left_kps  = (format_hand_keypoints(hand_map['Left'])
                 if 'Left'  in hand_map else zero_hand_keypoints())

    # --- Step 6: Determine handedness string --------------------------
    has_right = 'Right' in hand_map
    has_left  = 'Left'  in hand_map
    if has_right and has_left:
        handedness = "both"
    elif has_right:
        handedness = "right"
    else:
        handedness = "left"

    return image_name, handedness, right_kps, left_kps


print("Single-image processing function defined.")


Single-image processing function defined.


In [7]:
# ============================================================
#  CELL 8 — MAIN EXTRACTION LOOP
# ============================================================

def run_extraction(
    input_folder,
    output_csv,
    bad_images_folder,
    log_file,
    model_path,
    label=None,              # Optional: derived from input_folder if not set
    max_num_hands=2,
    min_detection_conf=0.5,
    min_presence_conf=0.5,
    min_tracking_conf=0.5,
    resize_w=640,
    resize_h=360,
    progress_every=500,
    image_extensions=None,
):
    """
    Extract hand keypoints from every image in `input_folder` and
    append results to `output_csv`.

    Uses the mediapipe 0.10 Tasks API (HandLandmarker) with
    VisionRunningMode.IMAGE for independent per-frame inference.

    Output CSV row format (5 columns):
        image_name, label, handedness, right_hand_keypoints, left_hand_keypoints

    - handedness: "right", "left", or "both" (which hands were detected)
    - right/left_hand_keypoints: space-separated string of 63 floats:
        x0 y0 z0 x1 y1 z1 ... x20 y20 z20
    X and Y are in [0, 1] (MediaPipe-normalised to image dimensions).
    Z is depth relative to the wrist landmark (not normalised to image dimensions).
    If a hand is not detected, its slot is filled with 63 zeros.
    """
    if image_extensions is None:
        image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tiff'}

    # Derive label from the last folder component if not explicitly provided
    if label is None:
        label = os.path.basename(os.path.normpath(input_folder))

    # --- Setup --------------------------------------------------------
    logger = setup_logging(log_file)
    setup_output_dirs(output_csv, bad_images_folder)

    logger.info("=" * 60)
    logger.info(f"Extraction started — label: '{label}'")
    logger.info(f"Input  : {input_folder}")
    logger.info(f"Output : {output_csv}")
    logger.info(f"Model  : {model_path}")
    logger.info(f"Resize : {resize_w}×{resize_h}" if resize_w else "Resize : disabled")

    # --- Resume: load previously processed image names ----------------
    processed = load_processed_images(output_csv)
    logger.info(f"Resume : {len(processed)} image(s) already in CSV — will skip.")

    # --- Discover images ----------------------------------------------
    all_files = get_image_files(input_folder, image_extensions)
    total_found = len(all_files)
    logger.info(f"Found  : {total_found} image file(s) in folder.")

    to_process = [
        f for f in all_files
        if os.path.basename(f) not in processed
    ]
    remaining = len(to_process)
    logger.info(f"To process: {remaining} image(s) (after skipping already-done).")

    if not to_process:
        logger.info("Nothing new to process. Exiting.")
        return

    # --- Counters -----------------------------------------------------
    success_count = 0
    failed_count  = 0   # No hand detected
    error_count   = 0   # Corrupted / unexpected exception

    start_time = datetime.now()

    # --- Build HandLandmarker options (Tasks API) --------------------
    options = HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=model_path),
        running_mode=VisionRunningMode.IMAGE,   # each image is independent
        num_hands=max_num_hands,
        min_hand_detection_confidence=min_detection_conf,
        min_hand_presence_confidence=min_presence_conf,
        min_tracking_confidence=min_tracking_conf,
    )

    # --- Open CSV in append mode + start HandLandmarker --------------
    with open(output_csv, "a", newline="", encoding="utf-8") as csv_file:
        writer = csv.writer(csv_file)

        with HandLandmarker.create_from_options(options) as detector:

            for idx, image_path in enumerate(to_process, start=1):

                # --- Try to process the image -------------------------
                try:
                    result = process_image(
                        image_path, detector, logger,
                        resize_w, resize_h, bad_images_folder,
                    )
                except Exception as exc:
                    error_count += 1
                    logger.error(
                        f"Unexpected error on '{os.path.basename(image_path)}': {exc}"
                    )
                    continue  # keep going — one bad image must not stop the run

                # --- Write to CSV if detection succeeded --------------
                if result is None:
                    failed_count += 1
                else:
                    image_name, handedness, right_kps, left_kps = result

                    right_str = " ".join(str(v) for v in right_kps)  # 63 values — Right hand
                    left_str  = " ".join(str(v) for v in left_kps)   # 63 values — Left hand

                    # 5-column row: name | label | handedness | right | left
                    writer.writerow([image_name, label, handedness, right_str, left_str])
                    csv_file.flush()  # persist row immediately → safe for resume
                    success_count += 1

                # --- Periodic progress report -------------------------
                if idx % progress_every == 0:
                    elapsed = (datetime.now() - start_time).total_seconds()
                    rate = idx / elapsed if elapsed > 0 else 0.0
                    eta_s  = (remaining - idx) / rate if rate > 0 else 0.0
                    logger.info(
                        f"[{idx:>7}/{remaining}] "
                        f"OK={success_count}  MISS={failed_count}  ERR={error_count}  "
                        f"Speed={rate:.1f} img/s  ETA={eta_s/60:.1f} min"
                    )

    # --- Final summary ------------------------------------------------
    elapsed_total = (datetime.now() - start_time).total_seconds()
    logger.info("=" * 60)
    logger.info(f"DONE — label: '{label}'")
    logger.info(f"  Processed        : {remaining}")
    logger.info(f"  Successful       : {success_count}")
    logger.info(f"  No detection     : {failed_count}  (copied to bad_images/)")
    logger.info(f"  Errors           : {error_count}")
    logger.info(f"  Time elapsed     : {elapsed_total:.1f}s")
    logger.info(f"  Avg speed        : {remaining / elapsed_total:.1f} img/s"
                if elapsed_total > 0 else "  Avg speed        : N/A")
    logger.info(f"  Output CSV       : {output_csv}")
    logger.info(f"  Bad images folder: {bad_images_folder}")
    logger.info("=" * 60)


print("Main extraction loop function defined.")


Main extraction loop function defined.


In [8]:
# ============================================================
#  CELL 9 — RUN ALL EXTRACTIONS
#
#  Processes every job discovered by build_jobs() in Cell 4.
#  Results are APPENDED to their respective CSVs — re-running
#  is safe because already-processed images are skipped.
#
#  To run a single job only (e.g. job index 5):
#      job = EXTRACTION_JOBS[4]
#      run_extraction(**{k: job[k] for k in
#          ("input_folder","output_csv","bad_images_folder","log_file","label")},
#          model_path=MODEL_PATH, ...)
# ============================================================

print(f"Starting extraction — {len(EXTRACTION_JOBS)} job(s) total.\n")

for job_idx, job in enumerate(EXTRACTION_JOBS, start=1):
    print(f"{'='*60}")
    print(f"  Job {job_idx}/{len(EXTRACTION_JOBS)} — {job['label']}")
    print(f"{'='*60}")
    run_extraction(
        input_folder         = job["input_folder"],
        output_csv           = job["output_csv"],
        bad_images_folder    = job["bad_images_folder"],
        log_file             = job["log_file"],
        label                = job["label"],
        model_path           = MODEL_PATH,
        max_num_hands        = MAX_NUM_HANDS,
        min_detection_conf   = MIN_DETECTION_CONFIDENCE,
        min_presence_conf    = MIN_PRESENCE_CONFIDENCE,
        min_tracking_conf    = MIN_TRACKING_CONFIDENCE,
        resize_w             = RESIZE_WIDTH,
        resize_h             = RESIZE_HEIGHT,
        progress_every       = PROGRESS_EVERY,
        image_extensions     = IMAGE_EXTENSIONS,
    )

print("\nAll extraction jobs complete.")


Starting extraction — 1 job(s) total.

  Job 1/1 — ៍


2026-05-26 09:21:13 [INFO] ============================================================
2026-05-26 09:21:13 [INFO] Extraction started — label: '៍'
2026-05-26 09:21:13 [INFO] Input  : E:\Keypoint Extraction\៍
2026-05-26 09:21:13 [INFO] Output : E:\Keypoint Extraction\៍\Keypoint\៍_keypoints.csv
2026-05-26 09:21:13 [INFO] Model  : E:\Test-extract-keypoint-body-hand\hand_landmarker.task
2026-05-26 09:21:13 [INFO] Resize : disabled
2026-05-26 09:21:13 [INFO] Resume : 0 image(s) already in CSV — will skip.
2026-05-26 09:21:13 [INFO] Found  : 1000 image file(s) in folder.
2026-05-26 09:21:13 [INFO] To process: 1000 image(s) (after skipping already-done).
2026-05-26 09:22:39 [INFO] [    500/1000] OK=500  MISS=0  ERR=0  Speed=5.9 img/s  ETA=1.4 min
2026-05-26 09:24:12 [INFO] [   1000/1000] OK=1000  MISS=0  ERR=0  Speed=5.6 img/s  ETA=0.0 min
2026-05-26 09:24:13 [INFO] ============================================================
2026-05-26 09:24:13 [INFO] DONE — label: '៍'
2026-05-26 09:24:13 [I


All extraction jobs complete.


In [ ]:
# ============================================================
#  CELL 10 — OUTPUT VERIFICATION (Optional)
#  Run this cell after extraction to sanity-check any CSV.
# ============================================================

def verify_output(csv_path):
    """
    Load the output CSV and print a quick quality report:
      - Total rows and unique labels
      - Handedness distribution (right / left / both)
      - Sample rows
      - Keypoint vector length check (must be 63 per hand: 21 × x,y,z)
      - Range check: X and Y must be in [0, 1] (Z is depth — not range-checked)

    Expected CSV columns (5):
        image_name | label | handedness | right_hand_keypoints | left_hand_keypoints
    """
    if not os.path.exists(csv_path):
        print(f"[ERROR] File not found: {csv_path}")
        return

    rows = []
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        for row in reader:
            if row:
                rows.append(row)

    if not rows:
        print("[WARNING] CSV is empty.")
        return

    print(f"{'='*55}")
    print(f"  Output CSV: {csv_path}")
    print(f"{'='*55}")
    print(f"  Total rows     : {len(rows)}")

    labels = sorted(set(r[1] for r in rows if len(r) >= 2))
    print(f"  Unique labels  : {len(labels)}  →  {labels}")

    # --- Handedness distribution --------------------------------------
    from collections import Counter
    handedness_counts = Counter(r[2] for r in rows if len(r) >= 3)
    print(f"  Handedness     : right={handedness_counts.get('right', 0)}  "
          f"left={handedness_counts.get('left', 0)}  "
          f"both={handedness_counts.get('both', 0)}")

    # --- Vector length check -----------------------------------------
    errors = 0
    out_of_range_xy = 0
    for i, row in enumerate(rows):
        if len(row) < 5:
            print(f"  [ROW {i}] Too few columns: {len(row)} (expected 5)")
            errors += 1
            continue

        h1_vals = row[3].split()   # right hand (column 4)
        h2_vals = row[4].split()   # left hand  (column 5)

        if len(h1_vals) != 63:
            print(f"  [ROW {i}] Right hand has {len(h1_vals)} values (expected 63)")
            errors += 1
        if len(h2_vals) != 63:
            print(f"  [ROW {i}] Left hand has {len(h2_vals)} values (expected 63)")
            errors += 1

        # Check [0, 1] range for X and Y only.
        # Layout per landmark: x y z (indices 0,1,2 / 3,4,5 / ...).
        # X is at indices 0,3,6,...  Y is at indices 1,4,7,...
        # Z (depth) is at indices 2,5,8,... — not range-checked.
        for hand_vals in (h1_vals, h2_vals):
            for j, v in enumerate(hand_vals):
                if j % 3 != 2:   # skip every third value (Z)
                    fv = float(v)
                    if not (0.0 <= fv <= 1.0):
                        out_of_range_xy += 1

    print(f"  Column errors  : {errors}")
    print(f"  Out-of-range XY: {out_of_range_xy}  (X/Y values outside [0,1])")

    # --- Sample rows --------------------------------------------------
    print(f"\n  --- First 3 rows (truncated to first 9 values per hand) ---")
    for row in rows[:3]:
        h1_preview = " ".join(row[3].split()[:9]) + " ..." if len(row) > 3 else "N/A"
        h2_preview = " ".join(row[4].split()[:9]) + " ..." if len(row) > 4 else "N/A"
        print(f"    name       : {row[0]}")
        print(f"    label      : {row[1]}")
        print(f"    handedness : {row[2] if len(row) > 2 else 'N/A'}")
        print(f"    right hand : {h1_preview}")
        print(f"    left  hand : {h2_preview}")
        print()

    # --- Label distribution ------------------------------------------
    print(f"  --- Label distribution ---")
    label_counts = Counter(r[1] for r in rows if len(r) >= 2)
    for lbl, cnt in sorted(label_counts.items()):
        print(f"    {lbl:30s} : {cnt}")

    print(f"{'='*55}")
    print("  Verification complete.")
    print(f"{'='*55}\n")


# ---- Run verification on every job that has output -----------------
verified = 0
for job in EXTRACTION_JOBS:
    if os.path.exists(job["output_csv"]):
        verify_output(job["output_csv"])
        verified += 1
    else:
        print(f"[SKIP] Not yet created: {os.path.basename(job['output_csv'])}")

print(f"\nVerified {verified}/{len(EXTRACTION_JOBS)} CSV file(s).")


  Output CSV: E:\Finger Spelling Dataset\Extracted Keypoint\Extracted Keypoint\ក_main_keypoints.csv
  Total rows     : 700
  Unique labels  : 1  →  ['Main']
  Column errors  : 0
  Out-of-range   : 0  (values outside [0, 1])

  --- First 3 rows (truncated) ---
    name  : ក_Main_0001.png
    label : Main
    hand1 : 0.685575 0.122083 0.66269 0.207677 0.639508 0.355711 ...
    hand2 : 0.0 0.0 0.0 0.0 0.0 0.0 ...

    name  : ក_Main_0002.png
    label : Main
    hand1 : 0.679139 0.106281 0.664282 0.190267 0.641722 0.338827 ...
    hand2 : 0.0 0.0 0.0 0.0 0.0 0.0 ...

    name  : ក_Main_0003.png
    label : Main
    hand1 : 0.69515 0.15154 0.6745 0.199658 0.646308 0.334684 ...
    hand2 : 0.0 0.0 0.0 0.0 0.0 0.0 ...

  --- Label distribution ---
    Main                           : 700
  Verification complete.


: 